<a href="https://colab.research.google.com/github/nmatthews5/GithubAction/blob/master/Masters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the datasets
perf_df = pd.read_excel('/content/golf_player_dataset_full.xlsx')
odds_df = pd.read_excel('/content/golf_finishing_position_odds.xlsx')

print("Performance Data Sample:")
display(perf_df.head())
print("\nBetting Odds Sample:")
display(odds_df.head())

Performance Data Sample:


,Rank,Player,Fracas,Cut %,1st Rd Win %,Win %,Top 20 %,Top 10 %,Top 5 %
0,1,Jon Rahm,2.805,94.6,6.2,8.5,76.6,54.8,35.7
1,2,Matt Fitzpatrick,2.675,93.8,6.1,7.6,74.4,52.0,33.2
2,3,Ludvig Aberg,2.441,91.2,5.9,6.7,68.0,46.0,28.8
3,4,Bryson DeChambeau,2.444,91.0,5.7,6.0,67.1,44.8,27.5
4,5,Cameron Young,2.322,90.3,5.4,5.5,64.8,42.3,25.7



Betting Odds Sample:


,Player,Winner,Top 5,Top 10,Top 20,Top 30,Top 40
0,Scottie Scheffler,550,138,-137,-345,-670,-2000
1,Bryson DeChambeau,1000,250,120,-200,-500,-1000
2,Jon Rahm,1000,250,120,-200,-500,-1000
3,Rory McIlroy,1200,300,125,-175,-335,-835
4,Ludvig Åberg,1400,335,150,-175,-295,-715


### Betting Math Logic
To determine if a bet is worth it, we use this logic:
1. **Convert American Odds to Decimal Odds**:
   - If Odds (+) : `(Odds / 100) + 1`
   - If Odds (-) : `(100 / abs(Odds)) + 1`
2. **Implied Probability**: `1 / Decimal Odds`
3. **Value Identification**: If `Your Percentage > Implied Probability`, it is a positive value bet.

In [2]:
def american_to_implied(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

# We will need to merge these dataframes on the Player Name once we confirm column names.
print("Awaiting data inspection to perform the merge and EV calculation.")

Awaiting data inspection to perform the merge and EV calculation.


In [4]:
import unicodedata

def normalize_name(name):
    """Remove accents and lowercase for better matching."""
    if not isinstance(name, str): return name
    name = unicodedata.normalize('NFD', name).encode('ascii', 'ignore').decode('utf-8')
    return name.strip().lower()

# 1. Normalize Player names
perf_df['Player_norm'] = perf_df['Player'].apply(normalize_name)
odds_df['Player_norm'] = odds_df['Player'].apply(normalize_name)

# 2. Merge dataframes
merged_df = pd.merge(perf_df, odds_df, on='Player_norm', suffixes=('_perf', '_odds'))

# 3. Define all categories including Top 30 and Top 40
# Note: Assuming perf_df has these columns. If not, we will calculate what is available.
categories = {
    'Winner': 'Win %',
    'Top 5': 'Top 5 %',
    'Top 10': 'Top 10 %',
    'Top 20': 'Top 20 %',
    'Top 30': 'Top 30 %',
    'Top 40': 'Top 40 %'
}

results = []
for _, row in merged_df.iterrows():
    res = {'Player': row['Player_odds']}
    for odd_col, perf_col in categories.items():
        if odd_col in row and perf_col in row:
            implied = american_to_implied(row[odd_col]) * 100
            actual = row[perf_col]
            edge = actual - implied
            res[f'{odd_col}_Edge'] = round(edge, 2)
            res[f'{odd_col}_Value'] = "YES" if edge > 0 else "NO"
    results.append(res)

value_df = pd.DataFrame(results)

# Display summary for Top 5 and Top 20 as requested
print("Top Value Bets for Top 5 Finish:")
display(value_df[['Player', 'Top 5_Edge', 'Top 5_Value']].sort_values(by='Top 5_Edge', ascending=False).head(5))

print("\nTop Value Bets for Top 20 Finish:")
display(value_df[['Player', 'Top 20_Edge', 'Top 20_Value']].sort_values(by='Top 20_Edge', ascending=False).head(5))

Top Value Bets for Top 5 Finish:


,Player,Top 5_Edge,Top 5_Value
4,Collin Morikawa,12.01,YES
6,Robert MacIntyre,8.62,YES
3,Cameron Young,7.52,YES
0,Jon Rahm,7.13,YES
1,Ludvig Åberg,5.81,YES



Top Value Bets for Top 20 Finish:


,Player,Top 20_Edge,Top 20_Value
4,Collin Morikawa,24.16,YES
6,Robert MacIntyre,14.30,YES
10,Akshay Bhatia,13.14,YES
15,J.J. Spaun,10.69,YES
13,Patrick Reed,10.36,YES
